# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-cou

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
htt

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 8 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog / posts', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'},
  {'type': 'affiliations / Nebula site',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'press coverage',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'}]}

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 1M+ models
Trending on
this week
Models
Tongyi-MAI/Z-Image-Turbo
Updated
8 days ago
•
297k
•
2.8k
mistralai/Devstral-Small-2-24B-Instruct-2512
Updated
2 days ago
•
28.1k
•
379
microsoft/VibeVoice-Realtime-0.5B
Updated
4 days ago
•
159k
•
886
zai-org/AutoGLM-Phone-9B
Updated
7 days ago
•
51.6k
•
324
zai-org/GLM-4.6V-Flash
Updated
7 days ago
•
102k
•
460
Browse 1M+ models
Spaces
Running
on
Zero
621
Z Image Turbo
🖼
621
Generate stunning images from text prompts
Running
on
Zero
MCP
Featured
179
Qwen Image to LoRA
😭
179
Generate LoRA from a single image
Running
on
Zero
MCP
Feat

In [15]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [18]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\nTongyi-MAI/Z-Image-Turbo\nUpdated\n8 days ago\n•\n297k\n•\n2.8k\nmistralai/Devstral-Small-2-24B-Instruct-2512\nUpdated\n2 days ago\n•\n28.1k\n•\n379\nmicrosoft/VibeVoice-Realtime-0.5B\nUpdated\n4 days ago\n•\n159k\n•\n886\nzai-org/AutoGLM-Phone-9B\nUpdated\n7 days ago\n•\n51.6k\n•\n324\nzai-org/GLM-4.6V-Flash\nUpdated\n7 days ago\n•\n102k\n•\n460\nBrowse 1M+ models\nSpaces\nRunning\non\nZer

In [19]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [20]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the vibrant AI community building the future of machine learning. It is the collaborative platform where machine learning practitioners and organizations come together to create, share, and deploy machine learning models, datasets, and applications. Hosting over **1 million models** and **250,000+ datasets**, Hugging Face empowers developers, researchers, and companies to accelerate AI innovation across modalities including text, image, video, audio, and even 3D.

---

## Key Features

- **Extensive Model Hub**  
  Access and contribute to over 1 million open-source models on NLP, computer vision, speech, and more.

- **Rich Dataset Repository**  
  Discover and use 250,000+ datasets from diverse domains or upload your own for community collaboration.

- **Spaces for Applications**  
  Run interactive AI apps collaboratively with GPU-backed Spaces supporting demos and prototypes.

- **Community & Collaboration**  
  Connect with a global community of AI practitioners, share your work, and build your professional ML profile.

- **Multi-modality Support**  
  Work seamlessly with text, images, videos, audio, and 3D data using an end-to-end open-source stack.

---

## Enterprise Solutions

Hugging Face offers tailored **Enterprise and Team plans** designed to scale AI efforts in organizations with advanced features such as:

- **Enterprise-grade security** (SSO, granular access control, audit logs)
- **Private dataset and model storage** with additional private storage capacity
- **Advanced compute options** including ZeroGPU quota boosts for faster inference and hosting
- **Comprehensive analytics dashboard** to monitor model and dataset usage
- **Flexible billing** with annual commitments and managed budgets
- **Priority dedicated support** from Hugging Face experts

Starting at $20 per user/month for teams, and customizable enterprise contracts provide robust, secure, and scalable AI infrastructure suited to business needs.

---

## Pricing Plans

- **Free Tier:** Access many models, datasets, and community features at no cost.
- **Pro Account ($9/month):** Unlock 10× private storage, 20× inference credits, better GPU quota, Spaces Dev Mode & ZeroGPU hosting, and profile customization.
- **Team Plan ($20/user/month):** Designed for growing teams, includes enterprise security features and team management tools.
- **Enterprise:** Flexible contracts for large organizations, enhanced security, billing, and support.

---

## Company Culture

Hugging Face fosters an open, collaborative, and inclusive culture that empowers innovation in AI through:

- Open-source ethos: Building and maintaining leading community-driven tools and models.
- Transparency and sharing: Encouraging sharing of work, discussions, and building portfolios.
- Supportive community: A platform for learners, researchers, and professionals to engage and grow.
- Innovation at scale: Continuously expanding capabilities to support the latest AI technologies and modalities.

---

## Customers and Community

The Hugging Face platform is trusted by thousands of individual developers, researchers, and top AI organizations worldwide. It is the home of forward-thinking AI teams that require cutting-edge tools to create, deploy, and manage AI applications at scale—from startups to large enterprises in technology, healthcare, finance, and more.

---

## Careers & Opportunities

At Hugging Face, joining the team means contributing to the AI tools shaping the future. The company values:

- Passion for open source and AI
- Technical excellence and innovation
- Collaborative mindset and community engagement
- A drive to build scalable, user-friendly AI platforms

Prospective candidates can look forward to working in a fast-growing, dynamic environment deeply embedded in the AI ecosystem, with opportunities across engineering, research, community management, and enterprise solutions.

---

## Join Hugging Face

Whether you are an AI enthusiast, a data scientist, a researcher, or an enterprise looking to leverage state-of-the-art AI, Hugging Face provides the resources, community, and infrastructure to help you succeed.

- Explore **1M+ models**, **250k+ datasets**, and **400k+ applications**.
- Collaborate with a global network of AI practitioners.
- Scale your AI efforts with flexible plans for individuals and organizations.

**Sign up today and be part of the AI community building the future.**

Website: [huggingface.co](https://huggingface.co)

---

*Hugging Face – The AI community building the future.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [21]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Brochure

## About Hugging Face

Hugging Face is the leading AI community and collaboration platform dedicated to building the future of machine learning. It serves as a central hub where the global machine learning community collaborates, shares, and discovers models, datasets, and applications across multiple AI modalities including text, image, video, audio, and even 3D.

At its core, Hugging Face is committed to fostering an open and ethical AI future by empowering machine learning engineers, scientists, and users to connect, experiment, and advance open-source innovations.

---

## What Hugging Face Offers

### The Collaboration Platform
- **Model Hub:** Access over 1 million open-source ML models updated frequently by a vibrant community.
- **Dataset Repository:** Browse and contribute to 250,000+ datasets spanning diverse domains.
- **Spaces:** Host and deploy interactive machine learning apps and demos for public and private use.
- **Multi-Modality Coverage:** Supports a broad spectrum including NLP, computer vision, audio processing, and more.
- **Open Source Stack:** Accelerate ML development with comprehensive, community-driven libraries and tools.

### Enterprise Solutions
- Hugging Face provides paid compute resources and tailored enterprise-grade solutions to accelerate team productivity with secure, scalable infrastructure and the most advanced ML tooling.

---

## Community and Culture

- **Community-Focused:** Hugging Face thrives on a massive, fast-growing global community passionate about openness, collaboration, and cutting-edge AI research.
- **Inclusive and Ethical:** Committed to creating an ethical AI ecosystem that benefits everyone through transparency and shared knowledge.
- **Portfolio Building:** Users can build their personal ML profile by sharing projects, models, and applications, helping promote talent within the community.
- **Shared Learning:** Extensive documentation and community forums support knowledge exchange and skill development.

---

## Customers

Hugging Face serves a wide range of users:
- AI researchers and data scientists working on cutting-edge machine learning projects.
- Enterprises and teams requiring scalable, advanced ML tools and compute solutions.
- Developers and hobbyists experimenting, building, and deploying AI-powered applications.
- Organizations seeking open and ethical AI collaborations to accelerate innovation.

---

## Careers at Hugging Face

Join a pioneering team shaping the future of AI! Hugging Face offers exciting career opportunities for:
- Machine Learning Engineers
- Research Scientists
- Software Developers
- Community Managers
- Enterprise Solution Architects

At Hugging Face, employees are empowered to contribute to open-source projects, collaborate with a global talent pool, and make a meaningful impact in the AI community.

---

## Connect and Get Started

- Explore AI models, datasets, and applications: https://huggingface.co
- Sign up to contribute and build your ML portfolio.
- Engage with the community, access documentation, and deploy your own ML apps on Spaces.
- Learn about enterprise plans to scale up your team’s AI capabilities.

---

## Brand Identity

- Hugging Face is recognized by its vibrant yellow (#FFD21E), orange (#FF9D00), and gray (#6B7280) color palette.
- The brand symbolizes openness, friendliness, and innovation in AI technology.

---

**Hugging Face**  
The collaborative AI community building the future of machine learning.  
Join us to explore, create, and share the next wave of AI innovation!

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>